## Getting data

Datasets:
- House roll call data (JSON via API)
- Senate roll call data (XML file divided by session)
- Member data (JSON via API)
- Member bioguide (web scraping)
- Bill data (JSON via API)

In [ ]:
# API Function

import requests
import random
import json
import time
from requests.exceptions import HTTPError
from pathlib import Path
from datetime import datetime, timezone
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()  # loads variables from .env


# Base url for extraction
base_url = "https://api.congress.gov/v3"
API_KEY = os.getenv("API_KEY")

# API call
session = requests.Session()

def get_api_info(search_string, max_retries=5):
    url = f"{base_url}/{search_string}&api_key={API_KEY}"
    for i in range(max_retries):
        response = session.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait = (2 ** i) + random.uniform(0, 0.5)  # exponential backoff + jitter
            print(f"429 Rate limited. Waiting {wait:.1f}s...")
            time.sleep(wait)
        else:
            response.raise_for_status()
    raise Exception(f"Failed after {max_retries} retries: {url}")

# Save data as json file
def save_to_file(data, file_path):
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

output_path = "../data/silver"

retrieved_at = datetime.now(timezone.utc).isoformat()


In [ ]:
# House roll call data (JSON via API)

# House voting data for 119th congress (records = 420 as of 2/9/26)
pages = []

for offset in range(0, 420, 250):
    search_string = f"house-vote/119?format=json&offset={offset}&limit=250"
    page_data = get_api_info(search_string)
    pages.append(page_data)

final_payload = {
    "source": "https://api.congress.gov/",
    "retrieved_at": retrieved_at,
    "congress" : 119,
    "pages": pages
}

save_to_file(final_payload, f"{output_path}/house_rollcall_119.json")


In [ ]:
# Get House party vote total data (JSON via API)

# House voting data for 119th congress (votes for session 2 = 78 as of 3/1/26)
for session in range(1,3):
    pages = []

    if session == 1:
        num_of_votes = 362
    if session == 2:
        num_of_votes = 78

    for rollcall in range(1, num_of_votes+1):
        search_string = f"house-vote/119/1/{rollcall}?format=json"
        page_data = get_api_info(search_string)
        pages.append(page_data)

    # url: https://api.congress.gov/v3/house-vote/119/1/306?format=json&api_key=Y1rHJZnyeQR8VTyevR7cFJt0bjjzZn5nCDcvhj8f

    final_payload = {
        "source": "https://api.congress.gov/",
        "retrieved_at": retrieved_at,
        "congress" : 119,
        "pages": pages
    }

    save_to_file(final_payload, f"{output_path}/house_vote_party_totals_119_{session}.json")


In [ ]:
# Member data (JSON via API)

# List of members, bioguide, and basic info for 119th Congress (records = 538 as of 2/9/26)
pages = []

for offset in range(0, 541, 250):
    search_string = f"member?format=json&offset={offset}&limit=250&currentMember=true"
    page_data = get_api_info(search_string)
    pages.append(page_data)

final_payload = {
    "source": "https://api.congress.gov/",
    "retrieved_at": retrieved_at,
    "congress" : 119,
    "pages": pages
}

save_to_file(final_payload, f"{output_path}/members_119.json")

In [ ]:
# Bill data (JSON via API)

# Bill data for 119th Congress (records = 13192 as of 2/9/26)
pages = []

for offset in range(0, 13192, 250):
    search_string = f"bill/119?format=json&offset={offset}&limit=250"
    page_data = get_api_info(search_string)
    pages.append(page_data)

final_payload = {
    "source": "https://api.congress.gov/",
    "retrieved_at": retrieved_at,
    "congress" : 119,
    "pages": pages
}

save_to_file(final_payload, f"{output_path}/bills_119.json")


In [ ]:
# Sponsors, cosponsors, & policy area data (JSON via API)

with open("../data/silver/bills_119.json", "r", encoding="utf-8") as f:
    bills_data = json.load(f)

# Loop through bills and call APIs
def process_bill(bill_tuple):
    bill_type, bill_number = bill_tuple
    bill_id = f"{bill_type.upper()}{bill_number}_{congress}"

    # Bill detail
    search_string = f"bill/{congress}/{bill_type}/{bill_number}?format=json&offset=0&limit=250"
    bill_detail = get_api_info(search_string)
    bill_obj = bill_detail.get("bill", {})

    # Random small delay to avoid hitting rate limits
    time.sleep(random.uniform(0.1, 0.3))
    
    # Policy area
    policy_area = None
    if bill_obj.get("policyArea"):
        policy_area = bill_obj["policyArea"].get("name")

    # Sponsors
    sponsor_ids = [
        s.get("bioguideId")
        for s in bill_obj.get("sponsors", [])
        if s.get("bioguideId")
    ]

    # Cosponsors
    cosponsor_string = f"bill/{congress}/{bill_type}/{bill_number}/cosponsors?format=json&offset=0&limit=250"
    cosponsor_data = get_api_info(cosponsor_string)
    cosponsor_ids = [
        c.get("bioguideId")
        for c in cosponsor_data.get("cosponsors", [])
        if c.get("bioguideId")
    ]

    sponsorship_record = {
        "bill_id": bill_id,
        "bill_type": bill_type.upper(),
        "bill_number": bill_number,
        "sponsor_ids": sponsor_ids,
        "cosponsor_ids": cosponsor_ids
    }

    policy_record = {
        "bill_id": bill_id,
        "bill_type": bill_type.upper(),
        "bill_number": bill_number,
        "policy_area": policy_area
    }

    return sponsorship_record, policy_record

# 1. Load bills data
congress = 119
with open("../data/silver/bills_119.json", "r", encoding="utf-8") as f:
    bills_data = json.load(f)

# 2. Collect unique bills
unique_bills = set()
for page in bills_data.get("pages", []):
    for bill in page.get("bills", []):
        bill_type = bill.get("type")
        bill_number = bill.get("number")
        if bill_type and bill_number:
            unique_bills.add((bill_type.lower(), bill_number))
print("Unique bills collected")

# 3. Create Output containers
sponsorships_output = {"congress": congress, "bills": []}
policy_area_output = {"congress": congress, "bills": []}

# 4. Threaded processing
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_bill, b) for b in unique_bills]

    for future in as_completed(futures):
        sponsorship_record, policy_record = future.result()
        sponsorships_output["bills"].append(sponsorship_record)
        policy_area_output["bills"].append(policy_record)

# 5. Save files
save_to_file(sponsorships_output, "../data/silver/bills_sponsorships_119.json")
save_to_file(policy_area_output, "../data/silver/bills_policy_area_119.json")

Unique bills collected
429 Rate limited. Waiting 1.1s...
429 Rate limited. Waiting 1.2s...
429 Rate limited. Waiting 1.4s...
429 Rate limited. Waiting 1.2s...
429 Rate limited. Waiting 2.4s...
429 Rate limited. Waiting 1.4s...
429 Rate limited. Waiting 2.3s...
429 Rate limited. Waiting 2.3s...
429 Rate limited. Waiting 2.1s...
429 Rate limited. Waiting 2.4s...
429 Rate limited. Waiting 4.3s...
429 Rate limited. Waiting 4.1s...
429 Rate limited. Waiting 4.2s...
429 Rate limited. Waiting 4.2s...
429 Rate limited. Waiting 4.4s...
429 Rate limited. Waiting 8.5s...
429 Rate limited. Waiting 8.4s...
429 Rate limited. Waiting 8.2s...
429 Rate limited. Waiting 8.2s...
429 Rate limited. Waiting 8.0s...
429 Rate limited. Waiting 16.4s...
429 Rate limited. Waiting 16.3s...
429 Rate limited. Waiting 16.3s...
429 Rate limited. Waiting 16.5s...
429 Rate limited. Waiting 16.4s...
429 Rate limited. Waiting 1.2s...
429 Rate limited. Waiting 1.5s...
429 Rate limited. Waiting 1.0s...
429 Rate limited. Wa

: 

In [ ]:
# Senate roll call data (XML file divided by session)
import requests
import xmltodict
import json

# URLs
urls = []
for i in range(1,3):
    url = f"https://www.senate.gov/legislative/LIS/roll_call_lists/vote_menu_119_{i}.xml"
    urls.append(url)

print(urls)

combined_data = {}

for i, url in enumerate(urls, start=1):
    response = requests.get(url)
    response.raise_for_status()  # stop on HTTP errors

    # Parse XML to dict
    xml_dict = xmltodict.parse(response.text)

    # Store under a convenient key
    combined_data[f"vote_menu_119_{i}"] = xml_dict

# Write combined JSON
with open(f"{output_path}senate_rollcall_119.json", "w", encoding="utf-8") as f:
    json.dump(combined_data, f, indent=2, ensure_ascii=False)

print("Written senate_rollcall_119.json")




['https://www.senate.gov/legislative/LIS/roll_call_lists/vote_menu_119_1.xml', 'https://www.senate.gov/legislative/LIS/roll_call_lists/vote_menu_119_2.xml']
Written senate_roll_calls_119.json


In [ ]:
# Get vote records data - house & senate

import requests
import xml.etree.ElementTree as ET
from io import BytesIO
import os
import time
import json
from datetime import datetime, timezone


# Change directory
output_path = "../data/silver"

def getting_voting_data_xml_to_json(xml_base_url, json_output_name, chamber, congress, congress_session):
    print("Getting data for", json_output_name)

    session = requests.Session()
    session.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0 Safari/537.36")
    })

    rows = []
    delay_between_requests = 0.1
    if chamber == "Senate":
        delay_between_requests = 0.8
    max_backoff = 60

    # Loop to gather all valid URLs
    for i in range(1, 1000):  # safety upper bound
        url = xml_base_url.format(i)

        backoff = 2
        while True:
            r = session.get(url)
            if r.status_code == 403:
                print(f"403 received, backing off {backoff}s: {url}")
                time.sleep(backoff)
                backoff = min(backoff * 2, max_backoff)
                continue
            break

        if r.status_code == 404:
            print("404 received at: ", url)
            break

        r.raise_for_status()

        # Parse XML from memory
        try:
            tree = ET.parse(BytesIO(r.content))
        except ET.ParseError:
            break

        root = tree.getroot()

        # Construct data from XML
        if chamber == "House":
            source = "https://clerk.house.gov/Votes"
            roll_number = root.findtext(".//rollcall-num")
            bill = root.findtext(".//legis-num")
            date = root.findtext(".//action-date")

            for vote in root.findall(".//recorded-vote"):
                legislator = vote.find("legislator")
                rows.append({
                    "roll_number": roll_number,
                    "bill": bill,
                    "date": date,
                    "member": legislator.text,
                    "member_id": legislator.attrib.get("name-id"),
                    "vote": vote.findtext("vote")
                })

        elif chamber == "Senate":
            source = "https://www.senate.gov/legislative/votes_new.htm"
            roll_number = root.findtext(".//vote_number")
            bill = root.findtext(".//document/document_name")
            date = root.findtext(".//vote_date")

            for vote in root.findall(".//members/member"):
                rows.append({
                    "roll_number": roll_number,
                    "bill": bill,
                    "date": date,
                    "member": vote.findtext('member_full'),
                    "vote": vote.findtext("vote_cast")
                })

        time.sleep(delay_between_requests)


    # Generate timestamp
    retrieved_at = datetime.now(timezone.utc).isoformat()

    # Build final payload
    final_payload = {
        "source": source,
        "retrieved_at": retrieved_at,
        "congress" : congress,
        "session" : congress_session,
        "votes": rows   # your scraped records
    }

    # Save collected data as JSON
    with open(f"{output_path}/{json_output_name}", "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(rows)} records to silver/{json_output_name}")

    # Save collected data as JSON
    # with open(f"{output_path}/{json_output_name}", "w", encoding="utf-8") as f:
    #     json.dump(rows, f, ensure_ascii=False, indent=2)

    # print(f"Saved {len(rows)} records to silver/{json_output_name}")

congress=119
for i in range(1,3):
    session = i
    year = 1786+congress*2 + session

    senate_base_url = f"https://www.senate.gov/legislative/LIS/roll_call_votes/vote{congress}{session}/vote_{congress}_{session}_00{{:03d}}.xml"
    senate_output_json = f"senate_votes_{congress}_{session}.json"

    getting_voting_data_xml_to_json(senate_base_url, senate_output_json, "Senate", congress, session)

    house_base_url = f"https://clerk.house.gov/evs/{year}/roll{{:03d}}.xml"
    house_output_json = f"house_votes_{congress}_{session}.json"

    getting_voting_data_xml_to_json(house_base_url, house_output_json, "House", congress, session)

Getting data for senate_votes_119_1.json
Saved 65891 records to silver/senate_votes_119_1.json
Getting data for house_votes_119_1.json
404 received at:  https://clerk.house.gov/evs/2025/roll363.xml
Saved 156713 records to silver/house_votes_119_1.json
Getting data for senate_votes_119_2.json
Saved 4300 records to silver/senate_votes_119_2.json
Getting data for house_votes_119_2.json
404 received at:  https://clerk.house.gov/evs/2026/roll079.xml
Saved 33665 records to silver/house_votes_119_2.json


In [8]:
import json
import re
from collections import defaultdict

# Extracts party letter from member string
def extract_party(member_string):
    match = re.search(r"\(([A-Z])-", member_string)
    if match:
        return match.group(1)
    return None


def clean_and_tally_votes(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    session = int(input_file.split("_")[-1].replace(".json", ""))
    congress = str(data.get("congress"))
    session = str(data.get("session"))
    chamber = "S"  # Senate

    # Nested dictionary:
    # { (vote_id, party): {yes_count, no_count, present_count, not_voting_count} }
    tally = defaultdict(lambda: {
        "yes_count": 0,
        "no_count": 0,
        "present_count": 0,
        "not_voting_count": 0
    })

    valid_votes = {"Yea", "Nay", "Present", "Not Voting"}

    for record in data.get("votes", []):
        vote_type = record.get("vote")

        # Skip invalid vote types
        if vote_type not in valid_votes:
            continue

        roll_number = str(record.get("roll_number")).zfill(5)
        vote_id = f"roll_{chamber}{roll_number}_{congress}_{session}"

        party = extract_party(record.get("member", ""))

        # Skip if party cannot be extracted
        if party is None:
            continue

        key = (vote_id, party)

        if vote_type == "Yea":
            tally[key]["yes_count"] += 1
        elif vote_type == "Nay":
            tally[key]["no_count"] += 1
        elif vote_type == "Present":
            tally[key]["present_count"] += 1
        elif vote_type == "Not Voting":
            tally[key]["not_voting_count"] += 1

    # Convert aggregated results to list
    cleaned_records = []
    for (vote_id, party), counts in tally.items():
        cleaned_records.append({
            "vote_id": vote_id,
            "party": party,
            "yes_count": counts["yes_count"],
            "no_count": counts["no_count"],
            "present_count": counts["present_count"],
            "not_voting_count": counts["not_voting_count"]
        })

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_records, f, indent=4)

    print(f"Cleaned tally written to {output_file}")


clean_and_tally_votes(
    "../data/silver/senate_votes_119_1.json",
    "../data/silver/senate_vote_party_totals_119_1.json"
)

clean_and_tally_votes(
    "../data/silver/senate_votes_119_2.json",
    "../data/silver/senate_vote_party_totals_119_2.json"
)

Cleaned tally written to ../data/silver/senate_vote_party_totals_119_1.json
Cleaned tally written to ../data/silver/senate_vote_party_totals_119_2.json
